# IGF signaling pathway simulations

In [ ]:
# Load packages
from MScausality.causal_model.LVM import LVM
from MScausality.simulation.simulation import simulate_data
import MScausality.data_analysis.normalization as norm
import MScausality.data_analysis.gene_set as gs

import pyro
import pandas as pd
import numpy as np
from sklearn import linear_model

import networkx as nx
import y0
from y0.algorithm.simplify_latent import simplify_latent_dag
from y0.algorithm.identify import Identification, identify
from y0.dsl import P, Variable

import pickle
import copy

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

## Build network

In [ ]:
def build_igf_network(cell_confounder):
    """
    Create IGF graph in networkx
    
    cell_confounder : bool
        Whether to add in cell type as a confounder
    """
    graph = nx.DiGraph()

    ## Add edges
    graph.add_edge("EGF", "SOS")
    graph.add_edge("EGF", "PI3K")
    graph.add_edge("IGF", "SOS")
    graph.add_edge("IGF", "PI3K")
    graph.add_edge("SOS", "Ras")
    graph.add_edge("Ras", "PI3K")
    graph.add_edge("Ras", "Raf")
    graph.add_edge("PI3K", "Akt")
    graph.add_edge("Akt", "Raf")
    graph.add_edge("Raf", "Mek")
    graph.add_edge("Mek", "Erk")
    
    if cell_confounder:
        graph.add_edge("cell_type", "Ras")
        graph.add_edge("cell_type", "Raf")
        graph.add_edge("cell_type", "Mek")
        graph.add_edge("cell_type", "Erk")
    
    return graph

def build_admg(graph, cell_confounder=False, cell_latent=False):
    ## Define obs vs latent nodes
    all_nodes = ["SOS", "PI3K", "Ras", "Raf", "Akt", 
                 "Mek", "Erk", "EGF", "IGF"]
    obs_nodes = ["SOS", "PI3K", "Ras", "Raf", "Akt", 
                 "Mek", "Erk"]
    latent_nodes = ["EGF", "IGF"]
    
    ## Add in cell_type if included
    if cell_confounder:
        all_nodes.append("cell_type")
        if cell_latent:
            latent_nodes.append("cell_type")
        else:
            obs_nodes.append("cell_type")
        
    attrs = {node: (True if node not in obs_nodes and 
                    node != "\\n" else False) for node in all_nodes}

    nx.set_node_attributes(graph, attrs, name="hidden")
    
    ## Use y0 to build ADMG
    # simplified_graph = simplify_latent_dag(graph.copy(), "hidden")
    y0_graph = y0.graph.NxMixedGraph()
    y0_graph = y0_graph.from_latent_variable_dag(graph, "hidden")
    
    return y0_graph

In [ ]:
cell_type_graph = build_igf_network(cell_confounder=True)
bulk_graph = build_igf_network(cell_confounder=False)
y0_graph_bulk = build_admg(bulk_graph, cell_confounder=False)

In [ ]:
def plot_latent_graph(y0_graph, figure_size=(4, 3), title=None):

    ## Create new graph and specify color and shape of observed vs latent edges
    temp_g = nx.DiGraph()

    for d_edge in list(y0_graph.directed.edges):
        temp_g.add_edge(d_edge[0], d_edge[1], color="black", style='-', size=30)
    for u_edge in list(y0_graph.undirected.edges):
        if temp_g.has_edge(u_edge[0], u_edge[1]):
            temp_g.add_edge(u_edge[1], u_edge[0], color="red", style='--', size=1)
        else:
            temp_g.add_edge(u_edge[0], u_edge[1], color="red", style='--', size=1)

    ## Extract edge attributes
    pos = nx.random_layout(temp_g)
    edges = temp_g.edges()
    colors = [temp_g[u][v]['color'] for u, v in edges]
    styles = [temp_g[u][v]['style'] for u, v in edges]
    arrowsizes = [temp_g[u][v]['size'] for u, v in edges]

    ## Plot
    fig, ax = plt.subplots(figsize=figure_size)
    nx.draw_networkx_nodes(temp_g, pos=pos, node_size=1000, margins=[.1, .1], alpha=.7)
    nx.draw_networkx_labels(temp_g, pos=pos, font_weight='bold')
    nx.draw_networkx_edges(temp_g, pos=pos, ax=ax, connectionstyle='arc3, rad = 0.1',
                           edge_color=colors, width=3, style=styles, arrowsize=arrowsizes)
    if title is not None:
        ax.set_title(title)
    plt.show()
    
plot_latent_graph(y0_graph_bulk, figure_size=(12, 8))

## Ground truth invervention

Generate interventional data using the ground truth network for comparison.

In [ ]:
## Coefficients for relations
cell_coef = {'EGF': {'intercept': 19., "error": 1, "cell_type" : [3, 0, -3]},
              'IGF': {'intercept': 17., "error": 1, "cell_type" : [3, 0, -3]},
              'SOS': {'intercept': -4, "error": .5, 
                      'EGF': 0.6, 'IGF': 0.6, "cell_type" : [3, 0, -3]},
              'Ras': {'intercept': 10, "error": .5, 'SOS': .5, "cell_type" : [3, 0, -3]},
              'PI3K': {'intercept': 1.6, "error": .5, 
                       'EGF': .5, 'IGF': 0.5, 'Ras': .5, "cell_type" : [3, 0, -3]},
              'Akt': {'intercept': 2., "error": .5, 'PI3K': 0.75, "cell_type" : [3, 0, -3]},
              'Raf': {'intercept': 10, "error": .5,
                      'Ras': 0.8, 'Akt': -.4, "cell_type" : [-2, 0, 2]},
              'Mek': {'intercept': 3., "error": .5, 'Raf': 0.75, "cell_type" : [-2, 0, 2]},
              'Erk': {'intercept': 4., "error": .5, 'Mek': 1.2, "cell_type" : [-2, 0, 2]}
             }

In [ ]:
def obs_gt_sim(coef, n, cell_type, n_cells=1):
    
    """
    Observational ground truth simulation of network
    """
    
    if cell_type:
        cell_type = np.repeat([i for i in range(n_cells)], n//n_cells)
        if len(cell_type) < n:
            cell_type = np.append(cell_type, n_cells-1)
            
    EGF = np.random.normal(coef["EGF"]["intercept"], 3, n)
    IGF = np.random.normal(coef["IGF"]["intercept"], 3, n)

    SOS = coef["SOS"]["intercept"] + coef["SOS"]["EGF"]*EGF + coef["SOS"]["IGF"]*IGF + np.random.normal(0, 1., n)
    Ras = coef["Ras"]["intercept"] + coef["Ras"]["SOS"]*SOS + np.array(coef["Ras"]["cell_type"])[cell_type] + np.random.normal(0, 1., n)
    PI3K = coef["PI3K"]["intercept"] + coef["PI3K"]["Ras"]*Ras + coef["PI3K"]["EGF"]*EGF + coef["PI3K"]["IGF"]*IGF + np.random.normal(0, 1., n)
    Akt = coef["Akt"]["intercept"] + coef["Akt"]["PI3K"]*PI3K + np.random.normal(0, 1., n)
    Raf = coef["Raf"]["intercept"] + coef["Raf"]["Ras"]*Ras + coef["Raf"]["Akt"]*Akt + np.array(coef["Raf"]["cell_type"])[cell_type] + np.random.normal(0, 1., n)
    Mek = coef["Mek"]["intercept"] + coef["Mek"]["Raf"]*Raf + np.array(coef["Mek"]["cell_type"])[cell_type] + np.random.normal(0, 1., n)
    Erk = coef["Erk"]["intercept"] + coef["Erk"]["Mek"]*Mek + np.array(coef["Erk"]["cell_type"])[cell_type] + np.random.normal(0, 1., n)
    
    return({"EGF" : EGF, "IGF" : IGF, "SOS" : SOS, 
            "Ras" : Ras, "PI3K": PI3K, "Akt" : Akt,
            "Raf" : Raf, "Mek" : Mek, "Erk" : Erk, "cell_type" : cell_type})

def int_gt_sim(coef, n, inc_cell_type, n_cells=1):

    """
    Ras interventional simulation of network (Ras=10 vs Ras=20)
    """
    
    if inc_cell_type:
        cell_type = np.repeat([i for i in range(n_cells)], n//n_cells)
        if len(cell_type) < n:
            cell_type = np.append(cell_type, n_cells-1)
    else:
        cell_type = [1]
            
    EGF = np.random.normal(coef["EGF"]["intercept"], coef["EGF"]["error"], n)
    IGF = np.random.normal(coef["IGF"]["intercept"], coef["IGF"]["error"], n)

    SOS = sum([coef["SOS"]["intercept"], coef["SOS"]["EGF"]*EGF, 
               coef["SOS"]["IGF"]*IGF, np.random.normal(0, coef["SOS"]["error"], n)])
    Ras = 15
    PI3K = sum([coef["PI3K"]["intercept"], coef["PI3K"]["Ras"]*Ras, 
                coef["PI3K"]["EGF"]*EGF, coef["PI3K"]["IGF"]*IGF,
                np.random.normal(0, coef["PI3K"]["error"], n)])
    Akt = sum([coef["Akt"]["intercept"], coef["Akt"]["PI3K"]*PI3K,
                np.random.normal(0, coef["Akt"]["error"], n)])
    Raf = sum([coef["Raf"]["intercept"], coef["Raf"]["Ras"]*Ras, 
               coef["Raf"]["Akt"]*Akt, 
               np.array(coef["Raf"]["cell_type"])[cell_type],
                 np.random.normal(0, coef["Raf"]["error"], n)])
    Mek = sum([coef["Mek"]["intercept"], coef["Mek"]["Raf"]*Raf,
                np.array(coef["Mek"]["cell_type"])[cell_type],
                  np.random.normal(0, coef["Mek"]["error"], n)])
    Erk_obs = sum([coef["Erk"]["intercept"], coef["Erk"]["Mek"]*Mek, 
                 np.array(coef["Erk"]["cell_type"])[cell_type],
                   np.random.normal(0, coef["Erk"]["error"], n)])
    
    EGF = np.random.normal(coef["EGF"]["intercept"], coef["EGF"]["error"], n)
    IGF = np.random.normal(coef["IGF"]["intercept"], coef["IGF"]["error"], n)

    SOS = sum([coef["SOS"]["intercept"], coef["SOS"]["EGF"]*EGF, 
               coef["SOS"]["IGF"]*IGF, np.random.normal(0, coef["SOS"]["error"], n)])
    Ras = 20
    PI3K = sum([coef["PI3K"]["intercept"], coef["PI3K"]["Ras"]*Ras, 
                coef["PI3K"]["EGF"]*EGF, coef["PI3K"]["IGF"]*IGF,
                np.random.normal(0, coef["PI3K"]["error"], n)])
    Akt = sum([coef["Akt"]["intercept"], coef["Akt"]["PI3K"]*PI3K,
                np.random.normal(0, coef["Akt"]["error"], n)])
    Raf = sum([coef["Raf"]["intercept"], coef["Raf"]["Ras"]*Ras, 
               coef["Raf"]["Akt"]*Akt, 
               np.array(coef["Raf"]["cell_type"])[cell_type],
                 np.random.normal(0, coef["Raf"]["error"], n)])
    Mek = sum([coef["Mek"]["intercept"], coef["Mek"]["Raf"]*Raf,
                np.array(coef["Mek"]["cell_type"])[cell_type],
                  np.random.normal(0, coef["Mek"]["error"], n)])
    Erk_int = sum([coef["Erk"]["intercept"], coef["Erk"]["Mek"]*Mek, 
                 np.array(coef["Erk"]["cell_type"])[cell_type],
                   np.random.normal(0, coef["Erk"]["error"], n)])
    
    int_data = [Erk_obs, Erk_int]
    if inc_cell_type:
        int_data.append(cell_type)
        
    return int_data

gt_int_cell = int_gt_sim(cell_coef, 1000000, inc_cell_type=False, n_cells=1)

In [ ]:
gt_int_cell

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

ax.hist(gt_int_cell[0], label="Ras=0", alpha=.75, bins=30)
ax.hist(gt_int_cell[1], label="Ras=2", alpha=.75, bins=30)
ax.legend()

# ax.hist(gt_int_cell[0][gt_int_cell[2] == 0], label="Ras=10", alpha=.75, bins=30)
# ax.hist(gt_int_cell[1][gt_int_cell[2] == 0], label="Ras=20", alpha=.75, bins=30)
# ax.legend()
# ax[1].hist(gt_int_cell[0][gt_int_cell[2] == 1], label="Ras=10", alpha=.75, bins=30)
# ax[1].hist(gt_int_cell[1][gt_int_cell[2] == 1], label="Ras=20", alpha=.75, bins=30)
# ax[2].hist(gt_int_cell[0][gt_int_cell[2] == 2], label="Ras=10", alpha=.75, bins=30)
# ax[2].hist(gt_int_cell[1][gt_int_cell[2] == 2], label="Ras=20", alpha=.75, bins=30)

ax.set_title("Ground Truth Invervention")
# ax.set_xlim(-5, 30)
# ax[1].set_xlim(-5, 30)
# ax[2].set_xlim(-5, 30)

In [ ]:
np.mean(gt_int_cell[1]) - np.mean(gt_int_cell[0])

## Cell Type - Bulk

### Simulate Data

In [ ]:
cell_sim = simulate_data(bulk_graph, coefficients=cell_coef, include_missing=True, 
                         cell_type=False, n_cells=1, n=200, seed=1)

Save feature level data to csvs to run data processing in MSstats in R. Wrapper for dataProcess functionality is being added to MScausality but is not currently available. For information on the R code to run please see the documentation for the dataProcess.py function (i.e., ?dataProcess)

In [ ]:
?dataProcess

In [ ]:
cell_sim["Feature_data"].to_csv("../../data/IGF_pathway/aug24_feature_data.csv", index=False)

In [ ]:
cell_sim_protein_data = pd.read_csv("../../data/IGF_pathway/aug24_protein_data.csv")
# cell_sim_protein_data.loc[:, "cell_type"] = cell_sim["Protein_data"]["cell_type"].astype(str)
cell_sim_protein_data.head()

In [ ]:
# cell_sim_protein_data = norm.normalize(cell_sim_protein_data)
cell_sim_protein_data = gs.prep_msstats_data(cell_sim_protein_data, gene_map=None, parse_gene=False)

In [ ]:
cell_sim_protein_data["Mek"].hist()

### Initial Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

plot_data = cell_sim_protein_data.dropna()
x = plot_data[plot_data["cell_type"] == "0"]["Ras"]
y = plot_data[plot_data["cell_type"] == "0"]["Erk"]

ax.scatter(x, y)
m, b = np.polyfit(x.values, y.values, 1)
ax.plot(x, m*x + b, color="red", lw=4)

ax.set_title("Regression Analysis", size=24)
ax.set_xlabel("Ras", size=24)
ax.set_ylabel("Erk", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

In [ ]:
fig, ax = plt.subplots(2,2,figsize=(10,6))

ras = cell_sim_protein_data["Ras"]
raf = cell_sim_protein_data["Raf"]
mek = cell_sim_protein_data["Mek"]
erk = cell_sim_protein_data["Erk"]

ax[0,0].scatter(ras, raf)
ax[0,1].scatter(raf, mek)
ax[1,0].scatter(mek, erk)

# ax.set_title("P(Erk | Ras)", size=24)
# ax.set_xlabel("Ras", size=24)
# ax.set_ylabel("Erk", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

### Run inference

In [ ]:
# pyro.clear_param_store()

lvm = LVM(cell_sim_protein_data.loc[:, 
        [i for i in cell_sim_protein_data.columns if i not in ["EGF", "IGF", "originalRUN"]]], 
          y0_graph_bulk)
lvm.prepare_graph()
lvm.prepare_data()
lvm.fit_model(num_steps=2000)

lvm.intervention({"Ras": 15}, "Erk")
first_int = lvm.intervention_samples
lvm.intervention({"Ras": 20}, "Erk")
second_int = lvm.intervention_samples

# lvm.intervention("Ras", "Erk", 10)
# first_int = lvm.intervention_samples

# lvm.intervention("Ras", "Erk", 20)
# second_int = lvm.intervention_samples

In [ ]:
lvm.parameters

In [ ]:
np.mean(np.array(second_int)) - np.mean(np.array(first_int))

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

adj = 0#np.mean(np.array(cell_sim_protein_data.loc[:, "Erk"].values))

ax.hist(np.array(first_int) - adj, bins=30, alpha=.75, label="Observational Data")
ax.axvline(np.mean(np.array(first_int)) - adj, color="darkblue", linestyle="dashed", lw=4)

ax2 = ax.twinx()
ax2.hist(np.array(second_int) - adj, bins=30, alpha=.75, label="Interventional Distribution", color="orange")
ax2.axvline(np.mean(np.array(second_int)) - adj, color="darkred", linestyle="dashed", lw=4)

# ax.legend(fontsize=20)
ax.set_title("ERK Interventional Distribution", size=24)
ax.set_xlabel("ERK - Log Fold Change", size=24)
# ax.set_ylabel("Erk", size=24)
ax.tick_params(axis='x', which='major', labelsize=16)
ax.set_yticklabels([])
ax2.set_yticklabels([])

# ax.set_xlim(-5,40)
# ax.set_ylim(0,1800)

## Cell Type - Single cell

### Analysis

In [ ]:
single_cell_lm = linear_model.LinearRegression()

fit_data = pd.concat([cell_sim_protein_data[["Ras", "Erk"]], 
               pd.get_dummies(cell_sim_protein_data["cell_type"].values)], 
              axis=1).dropna()
x = fit_data.drop(columns=["Erk"])
y = fit_data["Erk"]
single_cell_lm.fit(x, y)

bulk_lm = linear_model.LinearRegression()
x = cell_sim_protein_data.dropna()[["Ras"]]
y = cell_sim_protein_data.dropna()[["Erk"]]
bulk_lm.fit(x, y)

In [ ]:
## Plot
fig, ax = plt.subplots(1, 2, figsize=(14,6))

colors = np.where(cell_sim_protein_data["cell_type"] == "0", "Red", 
                  np.where(cell_sim_protein_data["cell_type"] == "1", "Orange", "Blue"))

ax[0].scatter(cell_sim_protein_data["Ras"], cell_sim_protein_data["Erk"])
ax[1].scatter(cell_sim_protein_data["Ras"], cell_sim_protein_data["Erk"], 
              c=colors, alpha=.5)

ax[0].plot(cell_sim_protein_data["Ras"], 
           bulk_lm.coef_[0]*cell_sim_protein_data["Ras"] + bulk_lm.intercept_[0], 
           color="red", lw=4)
ax[1].plot(cell_sim_protein_data["Ras"], 
           single_cell_lm.coef_[0]*cell_sim_protein_data["Ras"] + \
           single_cell_lm.intercept_ + single_cell_lm.coef_[1], 
           color="darkred", lw=4)
ax[1].plot(cell_sim_protein_data["Ras"], 
           single_cell_lm.coef_[0]*cell_sim_protein_data["Ras"] + \
           single_cell_lm.intercept_ + single_cell_lm.coef_[2], 
           color="darkOrange", lw=4)
ax[1].plot(cell_sim_protein_data["Ras"], 
           single_cell_lm.coef_[0]*cell_sim_protein_data["Ras"] + \
           single_cell_lm.intercept_ + single_cell_lm.coef_[3], 
           color="darkBlue", lw=4)

red_patch = mpatches.Patch(color='red', label='Cell Type 3')
orange_patch = mpatches.Patch(color='orange', label='Cell Type 2')
blue_patch = mpatches.Patch(color='blue', label='Cell Type 1')

ax[1].legend(handles=[blue_patch, orange_patch, red_patch], fontsize=16, loc="lower left", framealpha=1.)

ax[0].set_xlim(4,23)
ax[1].set_xlim(4,23)
ax[0].set_ylim(0,21)
ax[1].set_ylim(0,21)

ax[0].set_title("Bulk", fontsize=24)
ax[1].set_title("Single Cell", fontsize=24)

ax[0].set_xlabel("Ras", fontsize=18)
ax[1].set_xlabel("Ras", fontsize=18)

ax[0].set_ylabel("Erk", fontsize=18)
ax[1].set_ylabel("Erk", fontsize=18)

# plt.xticks(fontsize=16)
# plt.yticks(fontsize=16)

### Run inference

In [ ]:
datasets = {"Cell1" : cell_sim_protein_data[cell_sim_protein_data["cell_type"] == "0"].iloc[:,1:-1].reset_index(drop=True), 
            "Cell2" : cell_sim_protein_data[cell_sim_protein_data["cell_type"] == "1"].iloc[:,1:-1].reset_index(drop=True),
            "Cell3" : cell_sim_protein_data[cell_sim_protein_data["cell_type"] == "2"].iloc[:,1:-1].reset_index(drop=True)}

intervention_results = dict()

for name, data in datasets.items():
    pyro.clear_param_store()
    lvm = LVM(data, y0_graph_bulk)
    lvm.prepare_graph()
    lvm.prepare_data()
    lvm.fit_model(num_steps=4000)
    
    lvm.intervention("Ras", "Erk", 10)
    first_int = lvm.intervention_samples
    
    lvm.intervention("Ras", "Erk", 20)
    second_int = lvm.intervention_samples
    
    intervention_results[name] = [first_int, second_int, lvm]

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(16,5))

ax[0].hist(np.array(intervention_results["Cell1"][0]), label="Ras=10", alpha=.75, bins=20)
ax[0].axvline(np.mean(np.array(intervention_results["Cell1"][0])), color="darkblue", linestyle="dashed", lw=4)
ax[0].hist(np.array(intervention_results["Cell1"][1]), label="Ras=20", alpha=.75, bins=20)
ax[0].axvline(np.mean(np.array(intervention_results["Cell1"][1])), color="darkred", linestyle="dashed", lw=4)
ax[0].legend()

ax[1].hist(np.array(intervention_results["Cell2"][0]), label="Ras=10", alpha=.75, bins=20)
ax[1].axvline(np.mean(np.array(intervention_results["Cell2"][0])), color="darkblue", linestyle="dashed", lw=4)
ax[1].hist(np.array(intervention_results["Cell2"][1]), label="Ras=20", alpha=.75, bins=20)
ax[1].axvline(np.mean(np.array(intervention_results["Cell2"][1])), color="darkred", linestyle="dashed", lw=4)
ax[1].legend()

ax[2].hist(np.array(intervention_results["Cell3"][0]), label="Ras=10", alpha=.75, bins=20)
ax[2].axvline(np.mean(np.array(intervention_results["Cell3"][0])), color="darkblue", linestyle="dashed", lw=4)
ax[2].hist(np.array(intervention_results["Cell3"][1]), label="Ras=20", alpha=.75, bins=20)
ax[2].axvline(np.mean(np.array(intervention_results["Cell3"][1])), color="darkred", linestyle="dashed", lw=4)
ax[2].legend()

ax[0].set_title("Cell 1 Invervention", fontsize=16)
ax[1].set_title("Cell 2 Invervention", fontsize=16)
ax[2].set_title("Cell 3 Invervention", fontsize=16)
ax[0].set_xlim(-5, 30)
ax[1].set_xlim(-5, 30)
ax[2].set_xlim(-4, 30)
ax[0].set_xlabel("Erk - Log Intensity", size=14)
ax[1].set_xlabel("Erk - Log Intensity", size=14)
ax[2].set_xlabel("Erk - Log Intensity", size=14)
# ax.set_ylabel("Erk", size=24)

In [ ]:
np.mean(np.array(intervention_results["Cell1"][1])) - np.mean(np.array(intervention_results["Cell1"][0]))

In [ ]:
np.mean(np.array(intervention_results["Cell2"][1])) - np.mean(np.array(intervention_results["Cell2"][0]))

In [ ]:
np.mean(np.array(intervention_results["Cell3"][1])) - np.mean(np.array(intervention_results["Cell3"][0]))

## Low vs High Replicates

Compare causal effect estimation using different numbers of replicates.

### Simulate Data

In [ ]:
low_rep_data = simulate_data(bulk_graph, coefficients=cell_coef, include_missing=True, n=10, seed=1)
med_rep_data = simulate_data(bulk_graph, coefficients=cell_coef, include_missing=True, n=50, seed=0)
high_rep_data = simulate_data(bulk_graph, coefficients=cell_coef, include_missing=True, n=250, seed=0)
very_high_rep_data = simulate_data(bulk_graph, coefficients=cell_coef, include_missing=True, n=1000, seed=0)

In [ ]:
low_rep_data["Feature_data"].to_csv("../../data/IGF_pathway/low_rep_feature_data.csv", index=False)
med_rep_data["Feature_data"].to_csv("../../data/IGF_pathway/med_rep_feature_data.csv", index=False)
high_rep_data["Feature_data"].to_csv("../../data/IGF_pathway/high_rep_feature_data.csv", index=False)
very_high_rep_data["Feature_data"].to_csv("../../data/IGF_pathway/very_high_rep_feature_data.csv", index=False)

In [ ]:
low_rep_protein_data = pd.read_csv("../../data/IGF_pathway/low_rep_protein_data.csv")
med_rep_protein_data = pd.read_csv("../../data/IGF_pathway/med_rep_protein_data.csv")
high_rep_protein_data = pd.read_csv("../../data/IGF_pathway/high_rep_protein_data.csv")
very_high_protein_data = pd.read_csv("../../data/IGF_pathway/very_high_rep_protein_data.csv")

### Initial Analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,8))

ax[0,0].scatter(low_rep_protein_data["Ras"], low_rep_protein_data["Erk"])
ax[0,1].scatter(med_rep_protein_data["Ras"], med_rep_protein_data["Erk"])
ax[1,0].scatter(high_rep_protein_data["Ras"], high_rep_protein_data["Erk"])
ax[1,1].scatter(very_high_protein_data["Ras"], very_high_protein_data["Erk"])

ax[0,0].set_title("10 Cells", fontsize=18)
ax[0,1].set_title("50 Cells", fontsize=18)
ax[1,0].set_title("250 Cells", fontsize=18)
ax[1,1].set_title("1000 Cells", fontsize=18)

# ax[0,0].set_xlabel("Raf", fontsize=16)
# ax[0,1].set_xlabel("Raf", fontsize=16)
ax[1,0].set_xlabel("Ras", fontsize=16)
ax[1,1].set_xlabel("Ras", fontsize=16)

ax[0,0].set_ylabel("Erk", fontsize=16)
ax[0,1].set_ylabel("Erk", fontsize=16)
ax[1,0].set_ylabel("Erk", fontsize=16)
ax[1,1].set_ylabel("Erk", fontsize=16)

### Run inference

In [ ]:
low_rep_protein_data

In [ ]:
datasets = {"low" : low_rep_protein_data.iloc[:,1:], "med" : med_rep_protein_data.iloc[:,1:], 
            "high" : high_rep_protein_data.iloc[:,1:], "very_high" : very_high_protein_data.iloc[:,1:]}

intervention_results = dict()

for name, data in datasets.items():
    # data = pd.melt(data, id_vars='originalRUN', value_vars=['Akt','EGF','Erk',
    #                                                         'IGF','Mek','PI3K',
    #                                                         'Raf','Ras','SOS'],
    #                                                         var_name='Protein',
    #                                                         value_name='LogIntensities')
    # data = norm.normalize(data)
    # data = gs.prep_msstats_data(data, gene_map=None, parse_gene=False)
    pyro.clear_param_store()
    lvm = LVM(data, y0_graph_bulk)
    lvm.prepare_graph()
    lvm.prepare_data()
    lvm.fit_model(num_steps=2000)
    
    lvm.intervention({"Ras": 2}, "Erk")
    first_int = lvm.posterior_samples
    second_int = lvm.intervention_samples
    
    intervention_results[name] = [first_int, second_int, lvm]

### Compare to ground truth

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,8))

ax[0,0].hist(intervention_results["low"][0], label="Ras=10", alpha=.75, bins=30)
ax[0,0].hist(intervention_results["low"][1], label="Ras=20", alpha=.75, bins=30)
# ax[0,0].axvline(x=intervention_results["low"][0].mean(), color="darkblue", lw=3, linestyle="--")
# ax[0,0].axvline(x=intervention_results["low"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[0,0].axvline(x=intervention_results["low"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[0,0].axvline(x=intervention_results["low"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[0,0].legend()

ax[0,1].hist(intervention_results["med"][0], label="Ras=10", alpha=.75, bins=30)
ax[0,1].hist(intervention_results["med"][1], label="Ras=20", alpha=.75, bins=30)
# ax[0,1].axvline(x=intervention_results["med"][0].mean(), color="darkblue", lw=3, linestyle="--")
# ax[0,1].axvline(x=intervention_results["med"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[0,1].axvline(x=intervention_results["med"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[0,1].axvline(x=intervention_results["med"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[0,1].legend()

ax[1,0].hist(intervention_results["high"][0], label="Ras=10", alpha=.75, bins=30)
ax[1,0].hist(intervention_results["high"][1], label="Ras=20", alpha=.75, bins=30)
# ax[1,0].axvline(x=intervention_results["high"][0].mean(), color="darkblue", lw=3, linestyle="--")
# ax[1,0].axvline(x=intervention_results["high"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[1,0].axvline(x=intervention_results["high"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[1,0].axvline(x=intervention_results["high"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[1,0].legend()

ax[1,1].hist(intervention_results["very_high"][0], label="Ras=10", alpha=.75, bins=30)
ax[1,1].hist(intervention_results["very_high"][1], label="Ras=20", alpha=.75, bins=30)
# ax[1,1].axvline(x=intervention_results["very_high"][0].mean(), color="darkblue", lw=3, linestyle="--")
# ax[1,1].axvline(x=intervention_results["very_high"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[1,1].axvline(x=intervention_results["very_high"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[1,1].axvline(x=intervention_results["very_high"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[1,1].legend(fontsize=20)

np.mean(gt_int_cell[1]) - np.mean(gt_int_cell[0])

# ax[0,0].set_xlim(0,30)
# ax[0,1].set_xlim(0,30)
# ax[1,0].set_xlim(0,30)
# ax[1,1].set_xlim(0,30)

plt.legend()
ax[0,0].set_title("10 Cells", fontsize=18)
ax[0,1].set_title("50 Cells", fontsize=18)
ax[1,0].set_title("250 Cells", fontsize=18)
ax[1,1].set_title("1000 Cells", fontsize=18)

ax[1,0].set_xlabel("Erk - Log Intensity", size=16)
ax[1,1].set_xlabel("Erk - Log Intensity", size=16)

## Low vs High Replicates - Bulk

### Simulate data

In [ ]:
low_rep_bulk_data = simulate_data(cell_type_graph, coefficients=cell_coef, include_missing=True, 
                                  cell_type=True, n_cells=3, n=30, seed=0)
med_rep_bulk_data = simulate_data(cell_type_graph, coefficients=cell_coef, include_missing=True, 
                                  cell_type=True, n_cells=3, n=100, seed=0)
high_rep_bulk_data = simulate_data(cell_type_graph, coefficients=cell_coef, include_missing=True, 
                                   cell_type=True, n_cells=3, n=250, seed=0)
very_high_rep_bulk_data = simulate_data(cell_type_graph, coefficients=cell_coef, include_missing=True, 
                                        cell_type=True, n_cells=3, n=1000, seed=1)

In [ ]:
low_rep_bulk_data["Feature_data"].to_csv("../data/IGF_pathway/low_rep_bulk_feature_data.csv", index=False)
med_rep_bulk_data["Feature_data"].to_csv("../data/IGF_pathway/med_rep_bulk_feature_data.csv", index=False)
high_rep_bulk_data["Feature_data"].to_csv("../data/IGF_pathway/high_rep_bulk_feature_data.csv", index=False)
very_high_rep_bulk_data["Feature_data"].to_csv("../data/IGF_pathway/very_high_rep_bulk_feature_data.csv", index=False)

In [ ]:
low_rep_bulk_protein_data = pd.read_csv("../../data/IGF_pathway/low_rep_bulk_protein_data.csv")
med_rep_bulk_protein_data = pd.read_csv("../../data/IGF_pathway/med_rep_bulk_protein_data.csv")
high_rep_bulk_protein_data = pd.read_csv("../../data/IGF_pathway/high_rep_bulk_protein_data.csv")
very_high_bulk_protein_data = pd.read_csv("../../data/IGF_pathway/very_high_rep_bulk_protein_data.csv")

### Analysis

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,8))

ax[0,0].scatter(low_rep_bulk_protein_data["Ras"], low_rep_bulk_protein_data["Erk"])
ax[0,1].scatter(med_rep_bulk_protein_data["Ras"], med_rep_bulk_protein_data["Erk"])
ax[1,0].scatter(high_rep_bulk_protein_data["Ras"], high_rep_bulk_protein_data["Erk"])
ax[1,1].scatter(very_high_bulk_protein_data["Ras"], very_high_bulk_protein_data["Erk"])

ax[0,0].set_title("30 Bulk Samples", fontsize=18)
ax[0,1].set_title("100 Bulk Samples", fontsize=18)
ax[1,0].set_title("250 Bulk Samples", fontsize=18)
ax[1,1].set_title("1000 Bulk Samples", fontsize=18)

# ax[0,0].set_xlabel("Raf", fontsize=16)
# ax[0,1].set_xlabel("Raf", fontsize=16)
ax[1,0].set_xlabel("Ras", fontsize=16)
ax[1,1].set_xlabel("Ras", fontsize=16)

ax[0,0].set_ylabel("Erk", fontsize=16)
ax[0,1].set_ylabel("Erk", fontsize=16)
ax[1,0].set_ylabel("Erk", fontsize=16)
ax[1,1].set_ylabel("Erk", fontsize=16)

### Run inference

In [ ]:
datasets = {"low" : low_rep_bulk_protein_data.iloc[:,1:], "med" : med_rep_bulk_protein_data.iloc[:,1:], 
            "high" : high_rep_bulk_protein_data.iloc[:,1:], "very_high" : very_high_bulk_protein_data.iloc[:,1:]}

intervention_results = dict()

for name, data in datasets.items():
    pyro.clear_param_store()
    lvm = LVM(data, y0_graph_bulk)
    lvm.prepare_graph()
    lvm.prepare_data()
    lvm.fit_model(num_steps=2000)
    
    lvm.intervention("Ras", "Erk", 10)
    first_int = lvm.intervention_samples
    
    lvm.intervention("Ras", "Erk", 20)
    second_int = lvm.intervention_samples
    
    intervention_results[name] = [first_int, second_int, lvm]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,8))

ax[0,0].hist(intervention_results["low"][0], label="Ras=10", alpha=.75, bins=30)
ax[0,0].hist(intervention_results["low"][1], label="Ras=20", alpha=.75, bins=30)
ax[0,0].axvline(x=intervention_results["low"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[0,0].axvline(x=intervention_results["low"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[0,0].legend()

ax[0,1].hist(intervention_results["med"][0], label="Ras=10", alpha=.75, bins=30)
ax[0,1].hist(intervention_results["med"][1], label="Ras=20", alpha=.75, bins=30)
ax[0,1].axvline(x=intervention_results["med"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[0,1].axvline(x=intervention_results["med"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[0,1].legend()

ax[1,0].hist(intervention_results["high"][0], label="Ras=10", alpha=.75, bins=30)
ax[1,0].hist(intervention_results["high"][1], label="Ras=20", alpha=.75, bins=30)
ax[1,0].axvline(x=intervention_results["high"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[1,0].axvline(x=intervention_results["high"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[1,0].legend()

ax[1,1].hist(intervention_results["very_high"][0], label="Ras=10", alpha=.75, bins=30)
ax[1,1].hist(intervention_results["very_high"][1], label="Ras=20", alpha=.75, bins=30)
ax[1,1].axvline(x=intervention_results["very_high"][0].mean(), color="darkblue", lw=3, linestyle="--")
ax[1,1].axvline(x=intervention_results["very_high"][1].mean(), color="darkred", lw=3, linestyle="--")
ax[1,1].legend()

ax[0,0].set_xlim(-5,30)
ax[0,1].set_xlim(-5,30)
ax[1,0].set_xlim(-5,30)
ax[1,1].set_xlim(-5,30)

ax[0,0].set_title("10 Bulk Samples", fontsize=18)
ax[0,1].set_title("50 Bulk Samples", fontsize=18)
ax[1,0].set_title("250 Bulk Samples", fontsize=18)
ax[1,1].set_title("1000 Bulk Samples", fontsize=18)

ax[1,0].set_xlabel("Erk - Log Intensity", size=16)
ax[1,1].set_xlabel("Erk - Log Intensity", size=16)

## High dynamic range

In [ ]:
## Coefficients for relations
cell_coef_high_range = {'EGF': {'intercept': 18., "error": 8},
              'IGF': {'intercept': 17., "error": 8},
              'SOS': {'intercept': -4, "error": 1, 
                      'EGF': 0.6, 'IGF': 0.6,},
              'Ras': {'intercept': 5, "error": 1, 'SOS': .5, "cell_type" : [3, 0, -3]},
              'PI3K': {'intercept': 1.6, "error": 1, 
                       'EGF': .5, 'IGF': 0.5, 'Ras': .5,},
              'Akt': {'intercept': 2., "error": 1, 'PI3K': 0.75, },
              'Raf': {'intercept': 2, "error": 1,
                      'Ras': 0.8, 'Akt': -.4, "cell_type" : [-2, 0, 2]},
              'Mek': {'intercept': 3., "error": 1, 'Raf': 0.75, "cell_type" : [-2, 0, 2]},
              'Erk': {'intercept': 4., "error": 1, 'Mek': 1.2, "cell_type" : [-2, 0, 2]}
             }

In [ ]:
low_rep_range_data = simulate_data(cell_type_graph, coefficients=cell_coef_high_range, include_missing=True, 
                                  cell_type=True, n_cells=3, n=30, seed=0)
med_rep_range_data = simulate_data(cell_type_graph, coefficients=cell_coef_high_range, include_missing=True, 
                                  cell_type=True, n_cells=3, n=100, seed=0)
high_rep_range_data = simulate_data(cell_type_graph, coefficients=cell_coef_high_range, include_missing=True, 
                                   cell_type=True, n_cells=3, n=250, seed=0)
# very_high_rep_bulk_data = simulate_data(cell_type_graph, coefficients=cell_coef, include_missing=True, 
#                                         cell_type=True, n_cells=3, n=1000, seed=1)

In [ ]:
low_rep_range_data["Feature_data"].to_csv("../data/IGF_pathway/low_rep_range_feature_data.csv", index=False)
med_rep_range_data["Feature_data"].to_csv("../data/IGF_pathway/med_rep_range_feature_data.csv", index=False)
high_rep_range_data["Feature_data"].to_csv("../data/IGF_pathway/high_rep_range_feature_data.csv", index=False)

In [ ]:
low_rep_range_protein_data = pd.read_csv("../data/IGF_pathway/low_rep_range_protein_data.csv")
med_rep_range_protein_data = pd.read_csv("../data/IGF_pathway/med_rep_range_protein_data.csv")
high_rep_range_protein_data = pd.read_csv("../data/IGF_pathway/high_rep_range_protein_data.csv")

### Analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15,5))

ax[0].scatter(low_rep_range_protein_data["Ras"], low_rep_range_protein_data["Mek"])
ax[1].scatter(med_rep_range_protein_data["Ras"], med_rep_range_protein_data["Mek"])
ax[2].scatter(high_rep_range_protein_data["Ras"], high_rep_range_protein_data["Mek"])

ax[0].set_title("30 Replicates", fontsize=18)
ax[1].set_title("100 Replicates", fontsize=18)
ax[2].set_title("250 Replicates", fontsize=18)

# ax[0,0].s|et_xlabel("Raf", fontsize=16)
# ax[0,1].set_xlabel("Raf", fontsize=16)
ax[0].set_xlabel("Ras", fontsize=16)
ax[1].set_xlabel("Ras", fontsize=16)
ax[2].set_xlabel("Ras", fontsize=16)

                  
ax[0].set_ylabel("Mek", fontsize=16)
ax[1].set_ylabel("Mek", fontsize=16)
ax[2].set_ylabel("Mek", fontsize=16)

### Run inference

In [ ]:
datasets = {"low" : low_rep_range_protein_data, "med" : med_rep_range_protein_data, 
            "high" : high_rep_range_protein_data}

intervention_results = dict()

for name, data in datasets.items():
    scm = SCM(data, y0_graph_bulk)
    scm.prepare_scm_input()
    scm.fit_scm(num_samples=1000, warmup_steps=3000, num_chains=4)
    
    scm.intervention("Ras", "Erk", 10)
    first_int = scm.intervention_samples
    
    scm.intervention("Ras", "Erk", 20)
    second_int = scm.intervention_samples
    
    intervention_results[name] = [first_int, second_int, scm]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12,8))

ax[0,0].hist(intervention_results["low"][0], label="Ras=10", alpha=.75, bins=30)
ax[0,0].hist(intervention_results["low"][1], label="Ras=20", alpha=.75, bins=30)
# ax[0,0].axvline(x=np.mean(gt_int_data[0]), color="Blue", lw=2, linestyle="--", label="Ground Truth Ras=10")
# ax[0,0].axvline(x=np.mean(gt_int_data[1]), color="Red", lw=2, linestyle="--", label="Ground Truth Ras=20")
ax[0,0].legend()

ax[0,1].hist(intervention_results["med"][0], label="Ras=10", alpha=.75, bins=30)
ax[0,1].hist(intervention_results["med"][1], label="Ras=20", alpha=.75, bins=30)
# ax[0,1].axvline(x=np.mean(gt_int_data[0]), color="Blue", lw=2, linestyle="--", label="Ground Truth Ras=10")
# ax[0,1].axvline(x=np.mean(gt_int_data[1]), color="Red", lw=2, linestyle="--", label="Ground Truth Ras=20")
ax[0,1].legend()

ax[1,0].hist(intervention_results["high"][0], label="Ras=10", alpha=.75, bins=30)
ax[1,0].hist(intervention_results["high"][1], label="Ras=20", alpha=.75, bins=30)
# ax[1,0].axvline(x=np.mean(gt_int_data[0]), color="Blue", lw=2, linestyle="--", label="Ground Truth Ras=10")
# ax[1,0].axvline(x=np.mean(gt_int_data[1]), color="Red", lw=2, linestyle="--", label="Ground Truth Ras=20")
ax[1,0].legend()

# ax[1,1].hist(intervention_results["very_high"][0], label="Ras=10", alpha=.75, bins=30)
# ax[1,1].hist(intervention_results["very_high"][1], label="Ras=20", alpha=.75, bins=30)
# ax[1,1].axvline(x=np.mean(gt_int_data[0]), color="Blue", lw=2, linestyle="--", label="Ground Truth Ras=10")
# ax[1,1].axvline(x=np.mean(gt_int_data[1]), color="Red", lw=2, linestyle="--", label="Ground Truth Ras=20")
# ax[1,1].legend(fontsize=20)

ax[1,0].set_xlim(0,80)
ax[1,1].set_xlim(0,40)

plt.legend()
ax[0,0].set_title("10 Replicates", fontsize=18)
ax[0,1].set_title("50 Repalicates", fontsize=18)
ax[1,0].set_title("250 Replicates", fontsize=18)
ax[1,1].set_title("1000 Replicates", fontsize=18)

ax[1,0].set_xlabel("Log Intensity", fontsize=16)
ax[1,1].set_xlabel("Log Intensity", fontsize=16)

## Bad node measurements

Here we simulate a high error for Mek, which is along the causal path from Ras to Erk

In [ ]:
bad_node_data = simulate_data(cell_type_graph, coefficients=cell_coef, include_missing=True, 
                              add_error=True, error_node="Mek", n=250, seed=1)

In [ ]:
bad_node_data["Feature_data"].to_csv("../data/IGF_pathway/bad_node_feature_data.csv", index=False)

In [ ]:
bad_node_protein_data = pd.read_csv("../data/IGF_pathway/bad_node_protein_data.csv")

### Analysis

In [ ]:
single_cell_lm = linear_model.LinearRegression()

ras_erk_lm = linear_model.LinearRegression()
x = bad_node_protein_data.dropna()[["Ras"]]
y = bad_node_protein_data.dropna()[["Erk"]]
ras_erk_lm.fit(x, y)

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

x = bad_node_protein_data["Ras"]
y = bad_node_protein_data["Erk"]

ax.scatter(x, y)
ax.plot(bad_node_protein_data["Ras"], 
           ras_erk_lm.coef_[0]*bad_node_protein_data["Ras"] + ras_erk_lm.intercept_[0], 
           color="red", lw=4)

ax.set_title("P(Erk | Ras)", size=24)
ax.set_xlabel("Ras", size=24)
ax.set_ylabel("Erk", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

In [ ]:
single_cell_lm = linear_model.LinearRegression()

ras_raf_lm = linear_model.LinearRegression()
x = bad_node_protein_data.dropna()[["Ras"]]
y = bad_node_protein_data.dropna()[["Raf"]]
ras_raf_lm.fit(x, y)

raf_mek_lm = linear_model.LinearRegression()
x = bad_node_protein_data.dropna()[["Raf"]]
y = bad_node_protein_data.dropna()[["Mek"]]
raf_mek_lm.fit(x, y)

mek_erk_lm = linear_model.LinearRegression()
x = bad_node_protein_data.dropna()[["Mek"]]
y = bad_node_protein_data.dropna()[["Erk"]]
mek_erk_lm.fit(x, y)

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5))

ax[0].scatter(bad_node_protein_data["Ras"], bad_node_protein_data["Raf"])
ax[0].plot(bad_node_protein_data["Ras"], 
           ras_raf_lm.coef_[0]*bad_node_protein_data["Ras"] + ras_raf_lm.intercept_[0], 
           color="red", lw=4)

ax[1].scatter(bad_node_protein_data["Raf"], bad_node_protein_data["Mek"])
ax[1].plot(bad_node_protein_data["Raf"], 
           raf_mek_lm.coef_[0]*bad_node_protein_data["Raf"] + raf_mek_lm.intercept_[0], 
           color="red", lw=4)

ax[2].scatter(bad_node_protein_data["Mek"], bad_node_protein_data["Erk"])
ax[2].plot(bad_node_protein_data["Mek"], 
           mek_erk_lm.coef_[0]*bad_node_protein_data["Mek"] + mek_erk_lm.intercept_[0], 
           color="red", lw=4)

ax[0].set_title("Ras -> Raf", fontsize=18)
ax[1].set_title("Raf -> Mek", fontsize=18)
ax[2].set_title("Mek -> Erk", fontsize=18)

ax[0].set_xlabel("Ras", fontsize=16)                  
ax[0].set_ylabel("Raf", fontsize=16)

ax[1].set_xlabel("Raf", fontsize=16)                  
ax[1].set_ylabel("Mek", fontsize=16)

ax[2].set_xlabel("Mek", fontsize=16)                  
ax[2].set_ylabel("Erk", fontsize=16)

### Run inference

In [ ]:
scm = SCM(bad_node_protein_data, y0_graph_bulk)
scm.prepare_scm_input()
scm.fit_scm(num_samples=1000, warmup_steps=4000, num_chains=4)

scm.intervention("Ras", "Erk", 10)
first_int = scm.intervention_samples

scm.intervention("Ras", "Erk", 20)
second_int = scm.intervention_samples

In [ ]:
np.mean(np.array(second_int)) - np.mean(np.array(first_int))

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

ax.hist(np.array(first_int), bins=30, alpha=.75, label="Ras=10")
ax.hist(np.array(second_int), bins=30, alpha=.75, label="Ras=20")
ax.axvline(x=np.mean(first_int), color="darkblue", lw=3, linestyle="--")
ax.axvline(x=np.mean(second_int), color="darkred", lw=3, linestyle="--")

ax.legend(fontsize=20)
ax.set_title("Erk Interventional Distribution", size=24)
ax.set_xlabel("Erk - Log Intensity", size=24)
# ax.set_ylabel("Erk", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)


ax.set_xlim(0,25)

### Remove Mek

In [ ]:
y0_no_mek = copy.copy(y0_graph_bulk)
y0_no_mek.directed.remove_edge(Variable("Raf"), Variable("Mek"))
y0_no_mek.directed.remove_edge(Variable("Mek"), Variable("Erk"))
y0_no_mek.directed.add_edge(Variable("Raf"), Variable("Erk"))

In [ ]:
scm = SCM(bad_node_protein_data, y0_no_mek)
scm.prepare_scm_input()
scm.fit_scm(num_samples=1000, warmup_steps=4000, num_chains=4)

scm.intervention("Ras", "Erk", 10)
first_int = scm.intervention_samples

scm.intervention("Ras", "Erk", 20)
second_int = scm.intervention_samples

In [ ]:
np.mean(np.array(second_int)) - np.mean(np.array(first_int))

In [ ]:
fig, ax = plt.subplots(figsize=(10,6))

ax.hist(np.array(first_int), bins=30, alpha=.75, label="Ras=10")
ax.hist(np.array(second_int), bins=30, alpha=.75, label="Ras=20")
ax.axvline(x=np.mean(first_int), color="darkblue", lw=3, linestyle="--")
ax.axvline(x=np.mean(second_int), color="darkred", lw=3, linestyle="--")

ax.legend(fontsize=20)
ax.set_title("Erk Interventional Distribution", size=24)
ax.set_xlabel("Erk - Log Intensity", size=24)
# ax.set_ylabel("Erk", size=24)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)


ax.set_xlim(0,25)

## Missingness

### Simulate Data

In [ ]:
high_missing_data = simulate_data(bulk_graph, coefficients=cell_coef, include_missing=True, 
                                  mar_missing_param=.1, mnar_missing_param=[-3, .15], 
                                  n=250, seed=0)

In [ ]:
high_missing_data["Feature_data"].to_csv("../data/IGF_pathway/high_missing_feature_data.csv", index=False)

In [ ]:
high_missing_protein_data = pd.read_csv("../../data/IGF_pathway/high_missing_protein_data.csv")

### Initial Analysis

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(14,6))


mnar_thresh = [1 / (1 + np.exp(-3 + (.4 * i))) for i in range(0, 32)]
ax[0].plot(mnar_thresh)

mnar_thresh = [1 / (1 + np.exp(-3 + (.15 * i))) for i in range(0, 32)]
ax[1].plot(mnar_thresh)

ax[0].set_xlabel("Log Intensity", size=20)
ax[1].set_xlabel("Log Intensity", size=20)

ax[0].set_ylabel("Feaure Missing Probability", size=20)
ax[1].set_ylabel("Feaure Missing Probability", size=20)

ax[0].set_title("MNAR Curve - Low Probability", size=20)
ax[1].set_title("MNAR Curve - High Probability", size=20)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(14,6))

ax[0].bar(x = high_rep_protein_data.iloc[: ,1:].isnull().sum(axis = 0).index,
          height=high_rep_protein_data.iloc[: ,1:].isnull().sum(axis = 0) / len(high_rep_protein_data))

ax[1].bar(x = high_missing_protein_data.iloc[: ,1:].isnull().sum(axis = 0).index, 
       height=high_missing_protein_data.iloc[: ,1:].isnull().sum(axis = 0) / len(high_missing_protein_data))

ax[0].set_ylim(0,1)
ax[1].set_ylim(0,1)

ax[0].set_title("Low Missingness", size=20)
ax[1].set_title("High Missingness", size=20)

ax[0].set_xlabel("Gene", size=20)
ax[1].set_xlabel("Gene", size=20)

ax[0].set_ylabel("Percent Missing Replicates", size=20)
ax[1].set_ylabel("Percent Missing Replicates", size=20)

In [ ]:
low_missing_lm = linear_model.LinearRegression()
x = high_rep_protein_data.dropna()[["Raf"]]
y = high_rep_protein_data.dropna()[["Mek"]]
low_missing_lm.fit(x, y)

high_missing_lm = linear_model.LinearRegression()
x = high_missing_protein_data.dropna()[["Raf"]]
y = high_missing_protein_data.dropna()[["Mek"]]
high_missing_lm.fit(x, y)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,6))

ax[0].scatter(high_rep_protein_data["Raf"], high_rep_protein_data["Mek"])
ax[0].plot(high_rep_protein_data["Raf"], 
           low_missing_lm.coef_[0]*high_rep_protein_data["Raf"] + low_missing_lm.intercept_[0], 
           color="red", lw=4)

ax[1].scatter(high_missing_protein_data["Raf"], high_missing_protein_data["Mek"])
ax[1].plot(np.arange(3,8.5), 
           high_missing_lm.coef_[0]*np.arange(3,8.5) + high_missing_lm.intercept_[0], 
           color="red", lw=4)

ax[0].set_xlim(0,8)
ax[1].set_xlim(0,8)
ax[0].set_ylim(0,12.5)
ax[1].set_ylim(0,12.5)

ax[0].set_title("Low Missingness", size=20)
ax[1].set_title("High Missingness", size=20)

ax[0].set_xlabel("Raf", size=20)
ax[1].set_xlabel("Raf", size=20)

ax[0].set_ylabel("Mek", size=20)
ax[1].set_ylabel("Mek", size=20)

### Run inference

In [ ]:
pyro.clear_param_store()
lvm = LVM(high_rep_protein_data.iloc[:,1:], y0_graph_bulk)
lvm.prepare_graph()
lvm.prepare_data()
lvm.fit_model(num_steps=4000)

lvm.intervention("Ras", "Erk", 10)
first_int = lvm.intervention_samples

lvm.intervention("Ras", "Erk", 20)
second_int = lvm.intervention_samples

low_missing_intervention_results = [first_int, second_int, lvm]

low_missing_intervention_results[1].mean() - low_missing_intervention_results[0].mean()

In [ ]:
pyro.clear_param_store()
lvm = LVM(high_missing_protein_data.iloc[:,1:], y0_graph_bulk)
lvm.prepare_graph()
lvm.prepare_data()
lvm.fit_model(num_steps=5000)

lvm.intervention("Ras", "Erk", 10)
first_int = lvm.intervention_samples

lvm.intervention("Ras", "Erk", 20)
second_int = lvm.intervention_samples

high_missing_intervention_results = [first_int, second_int, lvm]

np.mean(np.array(high_missing_intervention_results[1])) - np.mean(np.array(high_missing_intervention_results[0]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,6))

ax[0].hist(low_missing_intervention_results[0], label="Ras=10", alpha=.75, bins=25)
ax[0].hist(low_missing_intervention_results[1], label="Ras=20", alpha=.75, bins=25)
ax[0].axvline(x=low_missing_intervention_results[0].mean(), color="Blue", lw=2, linestyle="--")
ax[0].axvline(x=low_missing_intervention_results[1].mean(), color="Red", lw=2, linestyle="--")
ax[0].legend()

ax[1].hist(high_missing_intervention_results[0], label="Ras=10", alpha=.75, bins=25)
ax[1].hist(high_missing_intervention_results[1], label="Ras=20", alpha=.75, bins=25)
ax[1].axvline(x=high_missing_intervention_results[0].mean(), color="Blue", lw=2, linestyle="--")
ax[1].axvline(x=high_missing_intervention_results[1].mean(), color="Red", lw=2, linestyle="--")
ax[1].legend()

ax[0].set_xlim(-10,40)
ax[1].set_xlim(-10,40)

plt.legend()

ax[0].set_title("Low Missing Interventional Distribution", size=20)
ax[0].set_xlabel("Erk - Log Intensity", size=20)

ax[1].set_title("High Missing Interventional Distribution", size=20)
ax[1].set_xlabel("Erk - Log Intensity", size=20)

# # ax.set_ylabel("Erk", size=24)
# plt.xticks(fontsize=16)
# plt.yticks(fontsize=16)

### Figure out why Pyro model is broken

In [ ]:
lvm.imputed_data

In [ ]:
mek_data = high_missing_intervention_results[2].imputed_data.loc[
    low_missing_intervention_results[2].imputed_data["protein"] == "Mek"]
raf_data = high_missing_intervention_results[2].imputed_data.loc[
    low_missing_intervention_results[2].imputed_data["protein"] == "Raf"]

mek_data.loc[:, "full_int"] = np.where(np.isnan(mek_data.loc[:, "intensity"]),
                                       mek_data.loc[:, "imp_mean"], 
                                       mek_data.loc[:, "intensity"])
raf_data.loc[:, "full_int"] = np.where(np.isnan(raf_data.loc[:, "intensity"]),
                                       raf_data.loc[:, "imp_mean"], 
                                       raf_data.loc[:, "intensity"])


low_miss_mek_data = low_missing_intervention_results[2].imputed_data.loc[
    low_missing_intervention_results[2].imputed_data["protein"] == "Mek"]
low_miss_raf_data = low_missing_intervention_results[2].imputed_data.loc[
    low_missing_intervention_results[2].imputed_data["protein"] == "Raf"]

low_miss_mek_data.loc[:, "full_int"] = np.where(
    np.isnan(low_miss_mek_data.loc[:, "intensity"]),
    low_miss_mek_data.loc[:, "imp_mean"], 
    low_miss_mek_data.loc[:, "intensity"])
low_miss_raf_data.loc[:, "full_int"] = np.where(
    np.isnan(low_miss_raf_data.loc[:, "intensity"]),
    low_miss_raf_data.loc[:, "imp_mean"], 
    low_miss_raf_data.loc[:, "intensity"])


In [ ]:
fig, ax = plt.subplots(1, 2)

ax[0].scatter(low_miss_raf_data["full_int"], low_miss_mek_data["full_int"])
ax[1].scatter(raf_data["full_int"], mek_data["full_int"])

ax[0].set_xlim(0,8)
ax[1].set_xlim(0,8)
ax[0].set_ylim(0,10.5)
ax[1].set_ylim(0,10.5)

In [ ]:
lvm.parameters

### Checkout simulated data - Comparison

Add in imputed values

In [ ]:
mek_imp = lvm.summary_stats["imp_Mek"]["mean"]
raf_imp = scm.summary_stats["imp_Raf"]["mean"]
mek_std = scm.summary_stats["imp_Mek"]["std"]
raf_std = scm.summary_stats["imp_Raf"]["std"]

## Add in missing data
high_missing_protein_data.loc[:, "imp_Mek"] = high_missing_protein_data.loc[:, "Mek"].isna()
high_missing_protein_data.loc[:, "imp_Raf"] = high_missing_protein_data.loc[:, "Raf"].isna()
high_missing_protein_data.loc[:, "full_Mek"] = high_missing_protein_data.loc[:, "Mek"]
high_missing_protein_data.loc[:, "full_Raf"] = high_missing_protein_data.loc[:, "Raf"]

high_missing_protein_data.loc[:, "Mek_std"] = np.nan
high_missing_protein_data.loc[:, "Raf_std"] = np.nan
high_missing_protein_data.loc[high_missing_protein_data["imp_Mek"], "Mek_std"] = mek_std
high_missing_protein_data.loc[high_missing_protein_data["imp_Raf"], "Raf_std"] = raf_std

high_missing_protein_data.loc[high_missing_protein_data["imp_Mek"], "full_Mek"] = mek_imp
high_missing_protein_data.loc[high_missing_protein_data["imp_Raf"], "full_Raf"] = raf_imp

mek_imp_high = intervention_results["high"][2].summary_stats["imp_Mek"]["mean"]
raf_imp_high = intervention_results["high"][2].summary_stats["imp_Raf"]["mean"]
mek_std_high = intervention_results["high"][2].summary_stats["imp_Mek"]["std"]
raf_std_high = intervention_results["high"][2].summary_stats["imp_Raf"]["std"]

## Add in missing data
high_rep_protein_data.loc[:, "imp_Mek"] = high_rep_protein_data.loc[:, "Mek"].isna()
high_rep_protein_data.loc[:, "imp_Raf"] = high_rep_protein_data.loc[:, "Raf"].isna()
high_rep_protein_data.loc[:, "full_Mek"] = high_rep_protein_data.loc[:, "Mek"]
high_rep_protein_data.loc[:, "full_Raf"] = high_rep_protein_data.loc[:, "Raf"]

high_rep_protein_data.loc[:, "Mek_std"] = np.nan
high_rep_protein_data.loc[:, "Raf_std"] = np.nan
high_rep_protein_data.loc[high_rep_protein_data["imp_Mek"], "Mek_std"] = mek_std_high
high_rep_protein_data.loc[high_rep_protein_data["imp_Raf"], "Raf_std"] = raf_std_high

high_rep_protein_data.loc[high_rep_protein_data["imp_Mek"], "full_Mek"] = mek_imp_high
high_rep_protein_data.loc[high_rep_protein_data["imp_Raf"], "full_Raf"] = raf_imp_high

Calculate regression lines

In [ ]:
low_missing_lm = linear_model.LinearRegression()
x = high_rep_protein_data.dropna()[["Raf"]]
y = high_rep_protein_data.dropna()[["Mek"]]
low_missing_lm.fit(x, y)

low_missing_imp_lm = linear_model.LinearRegression()
x = high_rep_protein_data[["full_Raf"]]
y = high_rep_protein_data[["full_Mek"]]
low_missing_imp_lm.fit(x, y)

high_missing_lm = linear_model.LinearRegression()
x = high_missing_protein_data.dropna()[["Raf"]]
y = high_missing_protein_data.dropna()[["Mek"]]
high_missing_lm.fit(x, y)

high_missing_imp_lm = linear_model.LinearRegression()
x = high_missing_protein_data[["full_Raf"]]
y = high_missing_protein_data[["full_Mek"]]
high_missing_imp_lm.fit(x, y)

Plot

In [ ]:
## Plot
fig, ax = plt.subplots(2, 2, figsize=(14,8))

colors1 = np.where((high_rep_protein_data["imp_Mek"] == True) & \
                   (high_rep_protein_data["imp_Raf"] == True), "Red", 
                  np.where((high_rep_protein_data["imp_Raf"] == True) | \
                           (high_rep_protein_data["imp_Mek"] == True), "Orange", "Blue"))

colors2 = np.where((high_missing_protein_data["imp_Mek"] == True) & \
                   (high_missing_protein_data["imp_Raf"] == True), "Red", 
                  np.where((high_missing_protein_data["imp_Raf"] == True) | \
                           (high_missing_protein_data["imp_Mek"] == True), "Orange", "Blue"))

ax[0,0].scatter(high_rep_protein_data["Raf"], high_rep_protein_data["Mek"], c=colors1, alpha=.6)
ax[0,0].plot(high_rep_protein_data["Raf"], 
           low_missing_lm.coef_[0]*high_rep_protein_data["Raf"] + low_missing_lm.intercept_[0], 
           color="black", lw=4)

ax[1,0].scatter(high_rep_protein_data["full_Raf"], high_rep_protein_data["full_Mek"], 
              c=colors1, alpha=.6)
ax[1,0].plot(high_rep_protein_data["full_Raf"], 
           low_missing_imp_lm.coef_[0]*high_rep_protein_data["full_Raf"] + low_missing_imp_lm.intercept_[0], 
           color="black", lw=4)

ax[0,1].scatter(high_missing_protein_data["Raf"], high_missing_protein_data["Mek"], 
              c=colors2, alpha=.6)
ax[0,1].plot(np.arange(3,9), 
           high_missing_lm.coef_[0]*np.arange(3,9) + high_missing_lm.intercept_[0], 
           color="black", lw=4)

ax[1,1].scatter(high_missing_protein_data["full_Raf"], high_missing_protein_data["full_Mek"], 
              c=colors2, alpha=.6)
ax[1,1].plot(high_missing_protein_data["full_Raf"], 
           high_missing_imp_lm.coef_[0]*high_missing_protein_data["full_Raf"] + high_missing_imp_lm.intercept_[0], 
           color="black", lw=4)

red_patch = mpatches.Patch(color='red', label='Both Missing')
orange_patch = mpatches.Patch(color='orange', label='One Missing')
blue_patch = mpatches.Patch(color='blue', label='Observed')

ax[1,1].legend(handles=[red_patch, orange_patch, blue_patch])
ax[1,0].legend(handles=[red_patch, orange_patch, blue_patch])

ax[0,0].set_xlim(0,8)
ax[1,1].set_xlim(0,8)
ax[0,1].set_xlim(0,8)
ax[1,0].set_xlim(0,8)
ax[0,0].set_ylim(0,12.5)
ax[1,1].set_ylim(0,12.5)
ax[0,1].set_ylim(0,12.5)
ax[1,0].set_ylim(0,12.5)

ax[0,0].set_title("Low missingness - no imputation", size=20)
ax[1,1].set_title("High missingness - with imputation", size=20)
ax[1,0].set_title("Low missingness - with imputation", size=20)
ax[0,1].set_title("High missingness - no imputation", size=20)

# ax[0,0].set_xlabel("Raf")
# ax[0,1].set_xlabel("Raf")
ax[1,1].set_xlabel("Raf", size=20)
ax[1,0].set_xlabel("Raf", size=20)

ax[0,0].set_ylabel("Mek", size=20)
ax[0,1].set_ylabel("Mek", size=20)
ax[1,1].set_ylabel("Mek", size=20)
ax[1,0].set_ylabel("Mek", size=20)

### Checkout simulated data - only high missing

In [ ]:
mek_imp = scm.summary_stats["imp_Mek"]["mean"]
raf_imp = scm.summary_stats["imp_Raf"]["mean"]
mek_std = scm.summary_stats["imp_Mek"]["std"]
raf_std = scm.summary_stats["imp_Raf"]["std"]

## Add in missing data
high_missing_protein_data.loc[:, "imp_Mek"] = high_missing_protein_data.loc[:, "Mek"].isna()
high_missing_protein_data.loc[:, "imp_Raf"] = high_missing_protein_data.loc[:, "Raf"].isna()
high_missing_protein_data.loc[:, "full_Mek"] = high_missing_protein_data.loc[:, "Mek"]
high_missing_protein_data.loc[:, "full_Raf"] = high_missing_protein_data.loc[:, "Raf"]

high_missing_protein_data.loc[high_missing_protein_data["imp_Mek"], "full_Mek"] = mek_imp
high_missing_protein_data.loc[high_missing_protein_data["imp_Raf"], "full_Raf"] = raf_imp

In [ ]:
high_missing_protein_data

In [ ]:
high_missing_lm = linear_model.LinearRegression()
x = high_missing_protein_data.dropna()[["Raf"]]
y = high_missing_protein_data.dropna()[["Mek"]]
high_missing_lm.fit(x, y)

high_missing_imp_lm = linear_model.LinearRegression()
x = high_missing_protein_data[["full_Raf"]]
y = high_missing_protein_data[["full_Mek"]]
high_missing_imp_lm.fit(x, y)

In [ ]:
high_missing_imp_lm.coef_

In [ ]:
## Plot
fig, ax = plt.subplots(1, 2, figsize=(10,6))


colors2 = np.where((high_missing_protein_data["imp_Mek"] == True) & \
                   (high_missing_protein_data["imp_Raf"] == True), "Red", 
                  np.where((high_missing_protein_data["imp_Raf"] == True) | \
                           (high_missing_protein_data["imp_Mek"] == True), "Orange", "Blue"))


ax[0].scatter(high_missing_protein_data["Raf"], high_missing_protein_data["Mek"], 
              c=colors2, alpha=.6)
ax[0].plot(np.arange(3,9), 
           high_missing_lm.coef_[0]*np.arange(3,9) + high_missing_lm.intercept_[0], 
           color="black", lw=4)

ax[1].scatter(high_missing_protein_data["full_Raf"], high_missing_protein_data["full_Mek"], 
              c=colors2, alpha=.6)
ax[1].plot(high_missing_protein_data["full_Raf"], 
           high_missing_imp_lm.coef_[0]*high_missing_protein_data["full_Raf"] + high_missing_imp_lm.intercept_[0], 
           color="black", lw=4)

red_patch = mpatches.Patch(color='red', label='Both Missing')
orange_patch = mpatches.Patch(color='orange', label='One Missing')
blue_patch = mpatches.Patch(color='blue', label='Observed')

ax[1].legend(handles=[red_patch, orange_patch, blue_patch])

ax[1].set_xlim(0,8)
ax[0].set_xlim(0,8)
ax[1].set_ylim(0,12.5)
ax[0].set_ylim(0,12.5)

ax[1].set_title("High missingness - with imputation", size=20)
ax[0].set_title("High missingness - no imputation", size=20)

# ax[0,0].set_xlabel("Raf")
# ax[0,1].set_xlabel("Raf")
ax[1].set_xlabel("Raf", size=20)
ax[0].set_xlabel("Raf", size=20)

ax[0].set_ylabel("Mek", size=20)
ax[1].set_ylabel("Mek", size=20)

In [ ]:
high_missing_protein_data.loc[:, "Mek_std"] = np.nan
high_missing_protein_data.loc[:, "Raf_std"] = np.nan
high_missing_protein_data.loc[high_missing_protein_data["imp_Mek"], "Mek_std"] = mek_std
high_missing_protein_data.loc[high_missing_protein_data["imp_Raf"], "Raf_std"] = raf_std

In [ ]:
high_missing_protein_data

In [ ]:
fig, ax = plt.subplots(2,1)
sns.histplot(data=high_missing_protein_data, x="Mek_std", hue="imp_Raf", ax=ax[0])
sns.histplot(data=high_missing_protein_data, x="Raf_std", hue="imp_Mek", ax=ax[1])
ax[0].set_xlim(.6,1.3)
ax[1].set_xlim(.6,1.3)

In [ ]:
high_missing_protein_data.head()

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,6))
sns.histplot(data=high_missing_protein_data[-high_missing_protein_data["imp_Raf"]], x="Raf_error", ax=ax[0])
sns.histplot(data=high_missing_protein_data[high_missing_protein_data["imp_Raf"]], x="Raf_error", ax=ax[1])

In [ ]:
fig, ax = plt.subplots()
sns.histplot(data=high_missing_protein_data, x="Mek_error")

In [ ]:
high_missing_protein_data.loc[:, "True_Raf"] = high_missing_data["Protein_data"]["Raf"]
high_missing_protein_data.loc[:, "True_Mek"] = high_missing_data["Protein_data"]["Mek"]

In [ ]:
high_missing_protein_data.loc[:, "Raf_error"] = abs(high_missing_protein_data.loc[:, "True_Raf"] - high_missing_protein_data.loc[:, "full_Raf"])

high_missing_protein_data.loc[:, "missing_count"] = high_missing_protein_data.loc[:, ["Akt", "EGF", "Erk", "IGF", 
                                                                                     "Mek", "PI3K", "Raf", "Ras", "SOS"]].isna().sum(axis=1).values

fig, ax = plt.subplots(figsize=(8,6))
sns.histplot(data=high_missing_protein_data[high_missing_protein_data["imp_Raf"]], x="Raf_error", ax=ax)

fig, ax = plt.subplots(5,1, figsize=(8,16))
sns.histplot(data=high_missing_protein_data[(high_missing_protein_data["imp_Raf"]) & 
                                           (high_missing_protein_data["missing_count"] == 1)], x="Raf_error", ax=ax[0])
sns.histplot(data=high_missing_protein_data[(high_missing_protein_data["imp_Raf"]) & 
                                           (high_missing_protein_data["missing_count"] == 2)], x="Raf_error", ax=ax[1])
sns.histplot(data=high_missing_protein_data[(high_missing_protein_data["imp_Raf"]) & 
                                           (high_missing_protein_data["missing_count"] == 3)], x="Raf_error", ax=ax[2])
sns.histplot(data=high_missing_protein_data[(high_missing_protein_data["imp_Raf"]) & 
                                           (high_missing_protein_data["missing_count"] == 4)], x="Raf_error", ax=ax[3])
sns.histplot(data=high_missing_protein_data[(high_missing_protein_data["imp_Raf"]) & 
                                           (high_missing_protein_data["missing_count"] == 5)], x="Raf_error", ax=ax[4])
ax[0].set_xlim(0,4.5)
ax[1].set_xlim(0,4.5)
ax[2].set_xlim(0,4.5)
ax[3].set_xlim(0,4.5)
ax[4].set_xlim(0,4.5)

In [ ]:
high_missing_protein_data.loc[:, "Mek_error"] = abs(high_missing_protein_data.loc[:, "True_Mek"] - high_missing_protein_data.loc[:, "full_Mek"])

fig, ax = plt.subplots()
sns.histplot(data=high_missing_protein_data[high_missing_protein_data["imp_Mek"]], x="Mek_error")